# MobileNetV3-Large: phân loại rác 10 lớp
Notebook tuyến tính này dùng hai dataset đã gắn vào Kaggle, gọi trực tiếp các CLI của repository và chỉ đánh giá tập test một lần sau khi chọn checkpoint tốt nhất. Bật GPU trước khi chạy toàn bộ notebook.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = 'https://github.com/Nhan-create/waste-classifier-mobilenetv3.git'
BRANCH = 'feat/mobilenetv3-10-class'
WORK_ROOT = Path('/kaggle/working')
PROJECT_ROOT = WORK_ROOT / 'waste-classifier-mobilenetv3'
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPOSITORY, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project:', PROJECT_ROOT)

## 1. Xác định hai Kaggle input đã gắn và tạo cấu hình đã phân giải

In [ ]:
import yaml

PINNED_SOURCES = {
    'vn_trash': 'mrgetshjtdone/vn-trash-classification/versions/1',
    'garbage_v2': 'sumn2u/garbage-classification-v2/versions/12',
}
INPUT_ROOT = Path('/kaggle/input')

def attached_mount(fragment):
    matches = sorted(path for path in INPUT_ROOT.iterdir() if path.is_dir() and fragment in path.name.lower())
    if len(matches) != 1:
        raise RuntimeError(f'Expected one attached input containing {fragment!r}, found: {matches}')
    return matches[0]

def layout_root(mount, required_children):
    candidates = [mount, *(path for path in mount.rglob('*') if path.is_dir())]
    for candidate in candidates:
        if all((candidate / child).is_dir() for child in required_children):
            return candidate
    raise RuntimeError(f'Could not locate {required_children} below {mount}')

vn_root = layout_root(attached_mount('vn-trash-classification'), ('Train', 'Test'))
garbage_mount = attached_mount('garbage-classification-v2')
garbage_original = next((path.parent for path in [garbage_mount / 'original', *garbage_mount.rglob('original')] if path.is_dir()), None)
garbage_root = garbage_original or garbage_mount

config = yaml.safe_load(Path('configs/preprocessing_config.yaml').read_text(encoding='utf-8'))
config['dataset']['sources']['vn_trash']['root'] = str(vn_root)
config['dataset']['sources']['garbage_v2']['root'] = str(garbage_root)
resolved_preprocessing = Path('configs/preprocessing_config.kaggle.yaml')
resolved_preprocessing.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
for name, root in {'vn_trash': vn_root, 'garbage_v2': garbage_root}.items():
    count = sum(path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES for path in root.rglob('*'))
    print(name, PINNED_SOURCES[name], root, 'images_before_mapping=', count)

## 2. Hợp nhất, kiểm tra ảnh, loại trùng và chia train/val/test

In [ ]:
run_pipeline = [
    sys.executable, '-m', 'src.preprocessing.run_pipeline',
    '--config', str(resolved_preprocessing),
    '--raw-root', 'data/raw',
    '--metadata-root', 'data/metadata/v1',
    '--processed-root', 'data/processed/v1',
    '--report-root', 'outputs/data/v1',
]
subprocess.run(run_pipeline, check=True)

In [ ]:
import pandas as pd
from src.data.validation import validate_processed_dataset

manifest_path = Path('data/metadata/v1/split_manifest.csv')
processed_root = Path('data/processed/v1')
manifest = pd.read_csv(manifest_path)
print('images_after_mapping_and_audit=', len(manifest))
display(pd.crosstab(manifest['unified_label'], manifest['split'], margins=True))
validation = validate_processed_dataset(processed_root, manifest_path)
assert validation.is_valid, validation
print({'validation.is_valid': validation.is_valid, 'errors': validation.errors, 'warnings': validation.warnings})

## 3. Huấn luyện hai giai đoạn và chọn theo validation macro-F1

In [ ]:
training_dir = Path('/kaggle/working/training')
subprocess.run([
    sys.executable, '-m', 'src.training.train',
    '--data-root', str(processed_root),
    '--manifest', str(manifest_path),
    '--preprocessing-config', str(resolved_preprocessing),
    '--config', 'configs/model_config.yaml',
    '--output-dir', str(training_dir),
    '--device', 'cuda',
], check=True)

## 4. Đánh giá duy nhất checkpoint tốt nhất trên test

In [ ]:
evaluation_dir = Path('/kaggle/working/evaluation')
subprocess.run([
    sys.executable, '-m', 'src.evaluation.evaluate',
    '--checkpoint', str(training_dir / 'best.pt'),
    '--test-root', str(processed_root / 'test'),
    '--output-dir', str(evaluation_dir),
    '--device', 'cuda',
], check=True)

## 5. Đóng gói checkpoint, chỉ số, biểu đồ, manifest và cấu hình

In [ ]:
import zipfile

package_path = Path('/kaggle/working/waste-classifier-output.zip')
artifacts = {
    training_dir / 'best.pt': 'best.pt',
    evaluation_dir / 'metrics.json': 'metrics.json',
    evaluation_dir / 'per_class_metrics.csv': 'per_class_metrics.csv',
    evaluation_dir / 'confusion_matrix_raw.png': 'confusion_matrix_raw.png',
    evaluation_dir / 'confusion_matrix_normalized.png': 'confusion_matrix_normalized.png',
    manifest_path: 'split_manifest.csv',
    Path('data/metadata/label_mapping.csv'): 'label_mapping.csv',
    training_dir / 'resolved_config.yaml': 'model_resolved_config.yaml',
    resolved_preprocessing: 'preprocessing_resolved_config.yaml',
}
missing = [str(path) for path in artifacts if not path.is_file()]
assert not missing, f'Missing package artifacts: {missing}'
with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for source, archive_name in artifacts.items():
        archive.write(source, archive_name)
print(package_path, package_path.stat().st_size, 'bytes')